# Part B
**Members**: Daniel Andrade - Andrés Vega - Carlos Flores

We picked SQL injection as the vulnerability to defend.

In [6]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "seclab").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


### B1 — Negative model
#### 1. Coverage

In [7]:
from homework.hw2.andrade.defend.waf import confirm_coverage
from seclab.attack import print_summary

coverage_findings = confirm_coverage()
print_summary(coverage_findings)

sqli_blocked = [f for f in coverage_findings if f.payload.family == "sqli"]
assert all(f.confirmed for f in sqli_blocked), "every Part-A SQLi payload should be caught"
print(f"\nCoverage on Part-A payloads: {sum(f.confirmed for f in sqli_blocked)}/{len(sqli_blocked)}")


  3/4 payloads confirmed by the oracle

  family                 confirmed   tried  coverage
  --------------------------------------------------
  benign                         0       1       0%
  sqli                           3       3     100%

Coverage on Part-A payloads: 3/3


#### 2. Bypass the WAF (required)

Same intent as each Part-A payload, different syntax: a block comment
instead of `--`, a `>` tautology instead of `=`, and a `UNION`/`SELECT`
split across a literal newline so the single-line regex window never spans
it.

In [8]:
from homework.hw2.andrade.defend.waf import confirm_bypass

bypass_findings = confirm_bypass()
for f in bypass_findings:
    print(f.payload.intent, "-> confirmed:", f.confirmed, "|", f.evidence)

assert len(bypass_findings) >= 3 and all(f.confirmed for f in bypass_findings), (
    "need >=3 same-intent payloads that evade the WAF and still leak the canary"
)
print(f"\n{sum(f.confirmed for f in bypass_findings)}/{len(bypass_findings)} bypasses confirmed (WAF evaded, canary leaked).")

string-terminate via block comment -> confirmed: True | marker 'FLAG-sqli-' present
tautology using > instead of = -> confirmed: True | marker 'FLAG-sqli-' present
union/select split across a newline -> confirmed: True | marker 'FLAG-sqli-' present

3/3 bypasses confirmed (WAF evaded, canary leaked).


#### 3. FP cost on a benign corpus

We measure false positives on two sources: our own invented usernames, and
the course-provided `duel-2-web/benign_traffic.json` fixture (both of its
groups, `login_username` and `search_or_name`).

We report a *per-source* breakdown, because the combined rate is diluted by
the provided cases and the honest headline is the 18.8% on the invented,
adversarially-plausible usernames.

In [9]:
from homework.hw2.andrade.defend.waf import (
    confirm_benign_corpus,
    false_positive_breakdown,
    false_positive_rate,
)

fp_findings = confirm_benign_corpus()
rate, blocked = false_positive_rate(fp_findings)
print(f"Combined FP rate: {rate:.1%} ({len(blocked)}/{len(fp_findings)})")
print("\nPer-source breakdown:")
for src, b in sorted(false_positive_breakdown(fp_findings).items()):
    ex = f"  e.g. {b['examples']}" if b["examples"] else ""
    print(f"  {src:<34} {b['blocked']:>2}/{b['tried']:<2} = {b['rate']:>5.1%}{ex}")

print("\nLegitimate requests the WAF blocked:")
for item in blocked:
    print(" ", item)

Combined FP rate: 8.1% (3/37)

Per-source breakdown:
  benign: invented                    3/16 = 18.8%  e.g. ['anita--jones', 'name or id=1', "o'brien--legacy"]
  benign: provided/login_username     0/9  =  0.0%
  benign: provided/search_or_name     0/12 =  0.0%

Legitimate requests the WAF blocked:
  'anita--jones' [benign: invented]
  'name or id=1' [benign: invented]
  "o'brien--legacy" [benign: invented]


### B2 — Positive model

The fix replaces string-built SQL with a query that uses bound
parameters to travel to SQLite as data, never as text spliced
into the query, so no value can change what the query asks for.

In [10]:
from homework.hw2.andrade.defend.defend import (
    confirm_fix_holds,
    confirm_legit_login_still_works,
)

findings = confirm_fix_holds()
for f in findings:
    print(f.payload.family, repr(f.payload.text), "confirmed:", f.confirmed)

assert not any(f.confirmed for f in findings if f.payload.family == "sqli"), (
    "every Part-A SQLi payload must now FAIL against the parameterized query"
)
assert confirm_legit_login_still_works(), (
    "the fix must not break the real admin login -- denying everyone is not a fix"
)
print("\nAll Part-A SQLi payloads and B1 bypasses fail against the fix, and the legit login still works.")

sqli "' OR 1=1 -- " confirmed: False
sqli "admin' -- " confirmed: False
sqli "x' UNION SELECT user, secret FROM users--" confirmed: False
benign 'alice' confirmed: False
sqli-bypass "admin' /*" confirmed: False
sqli-bypass "' OR 1>0 /*" confirmed: False
sqli-bypass "x' UNION\nSELECT user, secret FROM users /*" confirmed: False

All Part-A SQLi payloads and B1 bypasses fail against the fix, and the legit login still works.


**The guarantee:** 

User input cannot alter the structure of the
`/login` query, for any input, **provided every query against this database
goes through bound parameters** (`?` placeholders) rather than string
interpolation. The condition matters, a single query elsewhere in the
codebase built by concatenation reopens the hole, no matter how well this
one is written.

**Why a WAF (B1) can't make this claim:** 

A blocklist recognizes known-bad
syntax. It has to enumerate the shapes an attack can take, and anything
it doesn't recognize gets through. 

Parameterizing the query removes the mechanism the attack needs, characters in the `user` field are no longer read as SQL syntax. The guarantee holds structurally, not because we found and blocked every payload we could think of.

### B1 vs B2

The blocklist raises cost (an attacker needs a bypass) but is bypassable
and noisy; the parameterized query gives a guarantee. 


| Axis | B1 — WAF blocklist (negative) | B2 — parameterized query (positive) |
|---|---|---|
| Part-A SQLi payloads stopped | 3/3 blocked | 3/3 neutralized |
| Same-intent bypasses that leak | 3/3 | 0/3 |
| Legit inputs wrongly blocked (axis 5) | 18.8% on invented (3/16); 0% on provided (0/21); 8.1% combined (3/37) | 0 |
| Guarantee | none: blocks listed syntax only | structural: input is always data |